# 01 — 2D Scan

Define a phase transition model, run the BubbleMaster 2D scan over γ and time,
and cache results for bootstrapping in notebook 02.

**Section 4** always writes both a CPU (GSL) and a GPU SLURM array script covering
all γ values — submit whichever backend you want (or both, to cross-check).
Set `RUN_LOCAL = True` if you'd rather run a quick scan synchronously in this
notebook instead (only sensible for a small `n_gamma`, no SLURM needed).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ptbuilder as pt
from ptbuilder.ic import _two_bubble_ic, _cutoff_times

## 1. Define the model

In [ ]:
# Choose potential and create PhysicsModel
lambda_bar = 0.069

# Set to True to run a quick scan synchronously in this notebook instead of
# submitting a SLURM job (only sensible for small n_gamma). SLURM scripts for
# both CPU and GPU are always generated in Section 4 regardless of this flag.
RUN_LOCAL = False

potential = pt.Phi4Potential(lambda_bar)
config    = pt.Config()          # uses default data/ directory and bin/ binaries
model     = pt.PhysicsModel(potential, config)

REPO_ROOT = Path(config.bubblemaster_bin).parent.parent

print(f"Model name:   {model.name}")
print(f"Results dir:  {model.results_dir}")
print(f"phi_false:    {potential.phi_false}")
print(f"phi_true:     {potential.phi_true}")
print(f"Delta V:      {potential.delta_V:.6f}")
print(f"RUN_LOCAL:    {RUN_LOCAL}")

## 2. Compute the instanton profile (disk-cached)

In [ ]:
profile = model.instanton

print(f"rin_0:   {profile.rin_0:.4f}")
print(f"rmid_0:  {profile.rmid_0:.4f}")
print(f"rout_0:  {profile.rout_0:.4f}")
print(f"wall width: {profile.wall_width:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(profile.R, profile.Phi)
plt.axvline(profile.rin_0,  ls='--', color='C1', label='r_in')
plt.axvline(profile.rmid_0, ls='--', color='C2', label='r_mid')
plt.axvline(profile.rout_0, ls='--', color='C3', label='r_out')
plt.xlabel('r')
plt.ylabel(r'$\phi$')
plt.title('Instanton profile')
plt.legend()
plt.tight_layout()

## 3. Define the scan grid

In [ ]:
n_gamma  = 32
n_t      = 32

# Time-grid max depends on lambda_bar — different potentials have different
# mass scales, hence different natural collision timescales. Matches the
# existing scan_cache.h5 data already on disk for each lb below; add an entry
# before scanning a new lambda_bar rather than guessing a default.
T_MAX_BY_LB = {
    0.069: 30.,
    0.84:  70.,
}
if lambda_bar not in T_MAX_BY_LB:
    raise ValueError(
        f"No known T_MAX for lambda_bar={lambda_bar}; add it to T_MAX_BY_LB "
        f"(check the natural collision timescale for this potential before "
        f"picking a value — don't just reuse an existing one)."
    )
T_MAX = T_MAX_BY_LB[lambda_bar]

gammas   = np.linspace(1., 8., n_gamma)
times    = np.linspace(1., T_MAX, n_t)

bm_params = pt.BubbleMasterParams(
    dz           = 0.005,
    n_w          = 128,
    n_k          = 51,
    how_often_ds = 5,
    baby_steps   = 100,
    cutoff_type  = 0,
)

scan_cfg = pt.ScanConfig(
    gammas    = gammas,
    times     = times,
    bm_params = bm_params,
)

print(f"lambda_bar: {lambda_bar}  (T_MAX={T_MAX})")
print(f"Gammas: {gammas[[0,-1]]}  ({n_gamma} points)")
print(f"Times:  {times[[0,-1]]}  ({n_t} points)")

## 4. Run the scan

**Local** (`RUN_LOCAL=True`): runs synchronously in this notebook; each (γ, model) pair
is cached in `data/<model>/scan_cache.h5` — re-running only computes missing entries.
Only sensible for a small `n_gamma`.

**SLURM** (always generated below, regardless of `RUN_LOCAL`): writes setup files and
two array scripts — one CPU (GSL, one γ per node, 128 cores each) and one GPU (one γ
per GPU, 4 γ per node). Submit whichever you want, then run *§ 4b* to load results.

In [ ]:
if RUN_LOCAL:
    # ------------------------------------------------------------------ Local
    config.check_binaries()
    scan_results = pt.run_scan(model, scan_cfg, config, save_amplitude=True)
    print(f"\nScan complete: {len(scan_results)} gamma values")

else:
    # ------------------------------------------------------------------ Timing preview
    # d and effective t_max per gamma.
    # Reference fit from lb=0.069 validation; adjust _g_ref/_t_ref for other models.
    _g_ref = np.array([3.873, 5.9])
    _t_ref = np.array([135.,  219.])
    _exp   = np.log(_t_ref[1]/_t_ref[0]) / np.log(_g_ref[1]/_g_ref[0])
    _A     = _t_ref[0] / _g_ref[0]**_exp

    t_max_global = float(times[-1])
    print(f"Power-law exponent: {_exp:.3f}")
    print(f"\n{'γ':>8}  {'d':>8}  {'t_nat':>8}  {'t_eff':>8}  {'est (s)':>9}  note")
    print('-' * 60)
    est_wall = []
    for gamma in gammas:
        _, _, d, _ = _two_bubble_ic(profile, gamma, bm_params.dz)
        _, _, _, t_nat = _cutoff_times(d, bm_params.cutoff_type, bm_params.t_0_scal,
                                       t_min_global=0.)
        t_eff  = max(t_nat, t_max_global)
        t_est  = _A * gamma**_exp * (t_eff / t_nat)
        note   = 'forced' if t_eff > t_nat + 0.01 else 'natural'
        est_wall.append(t_est)
        print(f"{gamma:8.4f}  {d:8.2f}  {t_nat:8.2f}  {t_eff:8.2f}  {t_est:9.0f}  {note}")
    print(f"\nExpected wall time (slowest): {max(est_wall)/60:.1f} min")
    print("Note: this fit's reference point isn't tied to a specific backend/version —")
    print("treat it as a rough order-of-magnitude guide, not a precise per-backend estimate.")

In [ ]:
# Write one setup file per gamma, plus separate CPU/GPU output dirs so both
# backends can be submitted (even simultaneously) without clobbering each
# other's result_*.h5 files.
scan_dir = model.results_dir / 'scan'
scan_dir.mkdir(parents=True, exist_ok=True)

setup_paths, out_dirs_cpu, out_dirs_gpu = [], [], []
for gamma in gammas:
    setup_path  = scan_dir / f'setup_gamma{gamma:.6g}.h5'
    out_dir_cpu = scan_dir / f'out_gamma{gamma:.6g}_cpu'
    out_dir_gpu = scan_dir / f'out_gamma{gamma:.6g}_gpu'
    out_dir_cpu.mkdir(parents=True, exist_ok=True)
    out_dir_gpu.mkdir(parents=True, exist_ok=True)
    pt.write_2d_setup(model, gamma=gamma, times=times,
                      path=setup_path, params=bm_params)
    setup_paths.append(str(setup_path.resolve()))
    out_dirs_cpu.append(str(out_dir_cpu.resolve()))
    out_dirs_gpu.append(str(out_dir_gpu.resolve()))

print(f"Wrote {len(setup_paths)} setup files to {scan_dir}")

In [ ]:
# SLURM settings — set walltime based on the timing preview above
WALLTIME_CPU = '18:00:00'  # full 32-timestep GSL scan can run long at high gamma/large d
WALLTIME_GPU = '01:00:00'
ACCOUNT_GPU  = 'm3166_g'
QOS_GPU      = 'regular'
QOS_CPU      = 'premium'

n_tasks = n_gamma
assert n_tasks % 4 == 0, f"n_gamma ({n_tasks}) must be divisible by 4 (GPUs per node)"

scripts_dir = REPO_ROOT / 'scripts'
logs_dir    = REPO_ROOT / 'logs'
scripts_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)

model_tag = model.name.replace(' ', '_')
setup_arr = ' '.join(f'"{p}"' for p in setup_paths)

# --- GPU: one gamma per GPU, 4 GPUs (tasks) per node ---
gpu_bin          = str(config.bubblemaster_gpu_bin.resolve())
gpu_out_arr      = ' '.join(f'"{o}"' for o in out_dirs_gpu)
gpu_n_nodes      = n_tasks // 4
gpu_wrapper_path = scripts_dir / f'scan_{model_tag}_gpu_wrapper.sh'
gpu_slurm_path   = scripts_dir / f'scan_{model_tag}_gpu.sh'

gpu_wrapper_path.write_text(f"""#!/bin/bash
SETUPS=({setup_arr})
OUTDIRS=({gpu_out_arr})

IDX=$SLURM_PROCID
/usr/bin/time --verbose {gpu_bin} "${{SETUPS[$IDX]}}" "${{OUTDIRS[$IDX]}}/" --save-amplitude
""")
gpu_wrapper_path.chmod(0o755)

gpu_slurm_path.write_text(f"""#!/bin/bash
#SBATCH --job-name=scan_{model_tag}_gpu
#SBATCH --output={logs_dir}/scan_{model_tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos={QOS_GPU}
#SBATCH -A {ACCOUNT_GPU}
#SBATCH --nodes={gpu_n_nodes}
#SBATCH --ntasks={n_tasks}
#SBATCH --ntasks-per-node=4
#SBATCH --gpus-per-task=1
#SBATCH --time={WALLTIME_GPU}

srun --ntasks={n_tasks} {gpu_wrapper_path}
""")

# --- CPU: one gamma per node, all 128 cores each (GSL can't usefully share a
# node across gammas — a single run already saturates all cores via OpenMP) ---
cpu_bin          = str(config.bubblemaster_bin.resolve())
cpu_out_arr      = ' '.join(f'"{o}"' for o in out_dirs_cpu)
cpu_n_nodes      = n_tasks
cpu_wrapper_path = scripts_dir / f'scan_{model_tag}_cpu_wrapper.sh'
cpu_slurm_path   = scripts_dir / f'scan_{model_tag}_cpu.sh'

cpu_wrapper_path.write_text(f"""#!/bin/bash
SETUPS=({setup_arr})
OUTDIRS=({cpu_out_arr})

export SLURM_CPU_BIND=cores
export OMP_PLACES=threads
export OMP_PROC_BIND=spread
export OMP_NUM_THREADS=128

IDX=$SLURM_PROCID
/usr/bin/time --verbose {cpu_bin} "${{SETUPS[$IDX]}}" "${{OUTDIRS[$IDX]}}/" --save-amplitude
""")
cpu_wrapper_path.chmod(0o755)

cpu_slurm_path.write_text(f"""#!/bin/bash
#SBATCH --job-name=scan_{model_tag}_cpu
#SBATCH --output={logs_dir}/scan_{model_tag}_cpu_%j.out
#SBATCH --constraint=cpu
#SBATCH --qos={QOS_CPU}
#SBATCH --nodes={cpu_n_nodes}
#SBATCH --ntasks={n_tasks}
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=128
#SBATCH --time={WALLTIME_CPU}

srun --ntasks={n_tasks} --cpus-per-task=128 {cpu_wrapper_path}
""")

print(f'CPU script: {cpu_slurm_path}   ({cpu_n_nodes} nodes, 1 gamma/node, 128 cores each)')
print(f'GPU script: {gpu_slurm_path}   ({gpu_n_nodes} nodes, 4 gamma/node, 1 GPU each)')
print(f'\nSubmit whichever backend you want (or both — separate out_dirs, no clash):')
print(f'  sbatch {cpu_slurm_path}')
print(f'  sbatch {gpu_slurm_path}')

## 4b. Load SLURM results  *(run after a submitted job completes)*

Reads each γ output directory (preferring CPU results if both backends were run for
a given γ, otherwise falling back to GPU) and merges into `scan_cache.h5`.
Skip when `RUN_LOCAL = True` — the cache is already populated by Section 4.

In [ ]:
if not RUN_LOCAL:
    from ptbuilder.scan import _save_scan_cache

    scan_results = {}
    missing      = []

    for gamma, out_dir_cpu, out_dir_gpu in zip(gammas, out_dirs_cpu, out_dirs_gpu):
        cpu_path = Path(out_dir_cpu)
        gpu_path = Path(out_dir_gpu)
        if list(cpu_path.glob('result_*.h5')):
            out_path = cpu_path
        elif list(gpu_path.glob('result_*.h5')):
            out_path = gpu_path
        else:
            missing.append(gamma)
            continue
        scan_results[float(gamma)] = pt.load_bm_result(out_path, gamma, times)

    print(f"Loaded: {len(scan_results)}/{n_gamma} gamma points")
    if missing:
        print(f"Missing output for γ = {[f'{g:.4g}' for g in missing]}")

    cache_path = model.results_dir / 'scan_cache.h5'
    _save_scan_cache(scan_results, cache_path)
    print(f"Cached → {cache_path}")

## 5. Inspect results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

i_gamma_show = len(gammas) // 2
gamma_show   = gammas[i_gamma_show]
res          = scan_results[float(gamma_show)]

# Cumulative spectrum at each time
ax = axes[0]
for i_t in range(n_t):
    ax.plot(res.w, res.spectrum[i_t], alpha=0.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('w')
ax.set_ylabel('spectrum')
ax.set_title(f'Cumulative spectrum  (\u03b3={gamma_show:.1f})')

# dE/dt heatmap
ax = axes[1]
dEdt_show = np.array([scan_results[float(g)].dEdt[i_gamma_show]
                       for g in gammas])
ax.imshow(dEdt_show, aspect='auto', origin='lower', cmap='afmhot')
ax.set_xlabel('k index')
ax.set_ylabel('gamma index')
ax.set_title('dE/dt(\u03b3, k)')

plt.tight_layout()